# 03 — Chạy thử PBE tuần hoàn (pilot)
Chạy thử trên vài cấu trúc đại diện (cách điện / bán dẫn / kim loại) để **chốt** basis, giả thế, k-point, lưới FFT, spin và ngưỡng hội tụ trước khi chạy cả tập ở bước 04.

Giả thế và basis là **GTH-PBE + MOLOPT-SR có sẵn trong PySCF**, không cần tải file. Quy tắc chọn: lấy giá trị thô nhất mà năng lượng/nguyên tử lệch không quá `TOL_MEV` so với giá trị mịn nhất trong dải quét.

**Đầu ra:** `data/processed/pbe_settings.json` (bước 04 đọc file này), `pbe_pilot_scans.csv`.

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf pymatgen

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd, time
from utils import dft_utils as dft, structure_utils as su

TOL_MEV = 1.0                              # meV/nguyên tử
KS_LIST = [0.8, 0.6, 0.45, 0.35]           # 1/Å; càng nhỏ càng nhiều k-point (đắt hơn nhiều)
PREC_LIST = [1e-5, 1e-6, 1e-7, 1e-8]       # quyết định mesh FFT
SCAN_BASIS = "gth-szv-molopt-sr"           # basis nhỏ để quét tham số cho rẻ
BASIS_CHOICE = "gth-dzvp-molopt-sr"        # basis dùng cho bước 04 (xem bảng thời gian ở mục 4)
SPIN_CHECK = True                          # so sánh KRKS và KUKS trên một kim loại không từ tính
PILOT_IDS = None                           # ví dụ ["mp-134", "mp-23155"]; None = tự chọn
CONV_TOL, CONV_GRAD, MAX_CYCLE = 1e-9, 1e-6, 200
GAP_INSUL = dft.DEFAULT_SETTINGS["gap_insul_eV"]

sel = pd.read_csv(f"{PROC}/selected.csv")

## 1. Chọn cấu trúc đại diện
Mỗi nhóm lấy cấu trúc nhỏ nhất (ít nguyên tử, ít nguyên tố) để pilot rẻ.

In [ ]:
if PILOT_IDS is None:
    pool = sel.sort_values(["n_atoms", "n_elements"])
    groups = {"cách điện (gap>2)": pool.band_gap_eV > 2,
              "bán dẫn (0.5-2)": pool.band_gap_eV.between(GAP_INSUL, 2),
              "kim loại": pool.band_gap_eV < GAP_INSUL}
    pilot = pd.concat([pool[m].head(1).assign(group=n) for n, m in groups.items() if m.any()])
else:
    pilot = sel[sel.material_id.isin(PILOT_IDS)].assign(group="chỉ định")
pilot_structs = {m: su.read_structure(f"{RAW}/cif/{m}.cif") for m in pilot.material_id}
print(pilot[["material_id", "formula_pretty", "n_atoms", "band_gap_eV", "group"]].to_string(index=False))

In [ ]:
base = {**dft.DEFAULT_SETTINGS, "conv_tol": CONV_TOL, "conv_tol_grad": CONV_GRAD, "max_cycle": MAX_CYCLE}
scan_rows = []

def run_case(row, scan, value, **over):
    s = {**base, **over}
    info, *_ = dft.run_scf(pilot_structs[row.material_id], row.band_gap_eV, s)
    scan_rows.append({"material_id": row.material_id, "formula": row.formula_pretty, "scan": scan, "value": value,
                      "n_atoms": row.n_atoms, "E_per_atom_Ha": info["e_tot_per_atom_Ha"], "nk": info["nk"],
                      "mesh": info["mesh"][0], "nao": info["nao"], "cycles": info["cycles"],
                      "converged": info["converged"], "electron_ok": info["electron_ok"], "wall_s": info["wall_s"]})
    print(f"  {row.formula_pretty:8s} {scan}={value}  nk={info['nk']:4d} mesh={info['mesh'][0]:3d} "
          f"conv={info['converged']} e_ok={info['electron_ok']} {info['wall_s']}s")

def coarsest_ok(df, scan, finest_is_min):
    """Giá trị thô nhất mà |ΔE| so với giá trị mịn nhất <= TOL; trả về theo từng cấu trúc."""
    out = {}
    for mid, g in df[df.scan == scan].groupby("material_id"):
        g = g.sort_values("value", ascending=finest_is_min)
        e_ref = g.iloc[0]["E_per_atom_Ha"]
        g = g.assign(dE_meV=(g["E_per_atom_Ha"] - e_ref).abs() * dft.HA2EV * 1e3)
        ok = g[g.dE_meV <= TOL_MEV]
        out[mid] = ok.iloc[-1]["value"] if len(ok) else g.iloc[0]["value"]
    return out

## 2. Quét k-point (basis nhỏ, precision 1e-6)

In [ ]:
for r in pilot.itertuples():
    for ks in KS_LIST:
        run_case(r, "kspacing", ks, basis=SCAN_BASIS, precision=1e-6, kspacing=ks)
scan = pd.DataFrame(scan_rows)
ks_each = coarsest_ok(scan, "kspacing", finest_is_min=False)   # finest = giá trị nhỏ nhất
KSPACING = float(min(ks_each.values()))                       # lấy cấu trúc đòi hỏi mịn nhất
print("\nkspacing đủ cho từng cấu trúc:", ks_each, "-> chọn", KSPACING)
if KSPACING == min(KS_LIST):
    print("CẢNH BÁO: chưa thấy hội tụ trong dải quét; cân nhắc thêm giá trị nhỏ hơn vào KS_LIST.")

## 3. Quét precision (lưới FFT) tại k-mesh thô nhất

In [ ]:
for r in pilot.itertuples():
    for p in PREC_LIST:
        run_case(r, "precision", p, basis=SCAN_BASIS, precision=p, kspacing=max(KS_LIST))
scan = pd.DataFrame(scan_rows)
pr_each = coarsest_ok(scan, "precision", finest_is_min=True)    # finest = giá trị nhỏ nhất
PRECISION = float(min(pr_each.values()))
print("\nprecision đủ cho từng cấu trúc:", pr_each, "-> chọn", PRECISION)

## 4. So sánh basis (chi phí) — năng lượng giữa các basis khác nhau không so sánh trực tiếp được
Bảng cho biết số hàm cơ sở (`nao`) và thời gian; chọn `BASIS_CHOICE` ở đầu notebook theo ngân sách tính toán.

In [ ]:
for r in pilot.itertuples():
    for b in ["gth-szv-molopt-sr", "gth-dzvp-molopt-sr"]:
        run_case(r, "basis", b, basis=b, precision=PRECISION, kspacing=max(KS_LIST))
scan = pd.DataFrame(scan_rows)
scan[scan.scan == "basis"][["formula", "value", "nao", "mesh", "nk", "cycles", "wall_s"]]

## 5. Spin
Kim loại không từ tính: KUKS phải cho cùng năng lượng với KRKS. Nếu khác biệt nằm trong dung sai thì chốt `spin=False`.

In [ ]:
SPIN = False
if SPIN_CHECK and (pilot.band_gap_eV < GAP_INSUL).any():
    r = next(pilot[pilot.band_gap_eV < GAP_INSUL].itertuples())
    for sp in (False, True):
        run_case(r, "spin", sp, basis=SCAN_BASIS, precision=1e-6, kspacing=max(KS_LIST), spin=sp)
    sc = pd.DataFrame(scan_rows)
    e = sc[(sc.scan == "spin") & (sc.material_id == r.material_id)].set_index("value")["E_per_atom_Ha"]
    d = abs(e[True] - e[False]) * dft.HA2EV * 1e3
    print(f"|E(UKS) - E(RKS)| = {d:.3f} meV/nguyên tử (dung sai {TOL_MEV})")
    SPIN = bool(d > TOL_MEV)
print("Chốt spin =", SPIN, "(nguyên tố từ tính đã bị loại ở bước 02 khi EXCLUDE_MAGNETIC=True)")
scan = pd.DataFrame(scan_rows)
scan.to_csv(f"{PROC}/pbe_pilot_scans.csv", index=False)

## 6. Đo chi phí với cấu hình đã chốt và ước lượng tổng thời gian cho bước 04
Chạy 2 vòng SCF ở k-mesh thô để đo giây/k-point/vòng, rồi ngoại suy cho toàn tập với `N_CYCLES` vòng. Khoảng giá trị rất rộng vì chi phí mỗi k-point khác nhau nhiều giữa các nguyên tố.

In [ ]:
N_CYCLES = 12
FINAL = {**base, "basis": BASIS_CHOICE, "precision": PRECISION, "kspacing": KSPACING, "spin": SPIN}
spk = {}
for r in pilot.itertuples():
    info, *_ = dft.run_scf(pilot_structs[r.material_id], r.band_gap_eV,
                           {**FINAL, "kspacing": 1.0, "max_cycle": 2})
    spk[r.material_id] = info["wall_s"] / (info["nk"] * 3)      # init + 2 vòng
    print(f"  {r.formula_pretty:8s} nk={info['nk']:3d} nao={info['nao']:3d} mesh={info['mesh'][0]:3d} "
          f"-> {spk[r.material_id]:.2f} s/k-point/vòng")
nks = []
for r in sel.itertuples():
    st = su.read_structure(f"{RAW}/cif/{r.material_id}.cif")
    nks.append(int(np.prod(dft.kmesh_from(st, KSPACING, even=r.band_gap_eV < GAP_INSUL))))
lo, hi = min(spk.values()), max(spk.values())
print(f"\nTổng k-point toàn tập: {sum(nks):,}  |  ước lượng 1 core: {sum(nks)*N_CYCLES*lo/3600:.1f} - {sum(nks)*N_CYCLES*hi/3600:.1f} giờ")
print("Nếu quá lớn: tăng KSPACING, hạ PRECISION, giảm MAX_ATOMS ở bước 02 hoặc dùng basis szv.")

In [ ]:
settings = {**FINAL, "tol_meV_per_atom": TOL_MEV}
json.dump({"settings": settings,
           "pilot_ids": list(pilot.material_id),
           "scan_ranges": {"kspacing": KS_LIST, "precision": PREC_LIST},
           "per_structure_choice": {"kspacing": ks_each, "precision": pr_each},
           "s_per_k_per_cycle": spk,
           "note": "Giả thế GTH-PBE + basis MOLOPT-SR của PySCF; chốt từ pilot, xem pbe_pilot_scans.csv"},
          open(f"{PROC}/pbe_settings.json", "w"), indent=2, ensure_ascii=False, default=float)
print(json.dumps(settings, indent=2))